# 05 — MMM with Meta Robyn (R)

> **Objective**: Fit Meta's Robyn MMM framework using evolutionary (Nevergrad) optimisation to generate a Pareto-optimal frontier of model solutions.

**Key Topics**:
- Nevergrad evolutionary optimisation for adstock + saturation hyperparameters
- Pareto frontier: decomp.rssd vs. NRMSE
- One-pager summary chart (Robyn output)
- Budget allocator output
- Exporting ROAS results for Python comparison

**Framework**: R — `Robyn` package  
**Kernel**: R (IRkernel — run `IRkernel::installspec()` to register)

---
> **⚠️ Note**: This notebook requires an **R kernel**. In Jupyter Lab:  
> Kernel → Change Kernel → R

## 0. Setup & Imports (R)

In [ ]:

import subprocess, os, sys, json, textwrap
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
from pathlib import Path

# ── Project root (parent of this notebook) ───────────────────────────────────
PROJ_ROOT = Path(__file__).parent.parent if "__file__" in dir() else Path.cwd().parent
# Ensure we always resolve to the marketing-mix-modelling root
if not (PROJ_ROOT / "data").exists():
    PROJ_ROOT = Path(r"C:\Users\BIPLOB GON\Google Drive\DS & Analytics\github_projects\2026\marketing-mix-modelling\marketing-mix-modelling")
os.chdir(PROJ_ROOT)
print("Working directory:", Path.cwd())

R_EXE = r"C:\Program Files\R\R-4.5.3\bin\Rscript.exe"
PYTHON_EXE = r"C:\Users\BIPLOB GON\.local\bin\python3.14.exe"
R_ENV = {**os.environ, "TMP": r"C:\Rtemp", "TEMP": r"C:\Rtemp",
         "R_LIBS_USER": "C:/Rlibs", "RETICULATE_PYTHON": PYTHON_EXE}

os.makedirs("outputs/figures/robyn", exist_ok=True)
os.makedirs("outputs/models", exist_ok=True)

# ── Verify R + Robyn ─────────────────────────────────────────────────────────
r_check = subprocess.run(
    [R_EXE, "-e",
     ".libPaths(c('C:/Rlibs',.libPaths())); "
     "cat('Robyn:', as.character(packageVersion('Robyn')), '\\n'); "
     "cat('reticulate:', as.character(packageVersion('reticulate')), '\\n')"],
    capture_output=True, text=True, env=R_ENV)
print(r_check.stdout.strip())
if r_check.returncode != 0:
    print("WARNING:", r_check.stderr[:500])

# ── Verify nevergrad ─────────────────────────────────────────────────────────
ng_check = subprocess.run(
    [PYTHON_EXE, "-c", "import nevergrad; print('nevergrad:', nevergrad.__version__)"],
    capture_output=True, text=True)
print(ng_check.stdout.strip())

print("\n✓ Environment ready" if r_check.returncode == 0 and ng_check.returncode == 0
      else "\n✗ Environment issues detected — check above messages")


## 1. Load Data

In [ ]:

# Load raw data and aggregate to national weekly level
df_raw = pd.read_csv("data/raw/synthetic_mmm_weekly_india.csv")
df_raw["Week"] = pd.to_datetime(df_raw["Week"])

agg = {
    "Sales_Value":             "sum",
    "TV_Impressions":          "sum",
    "YouTube_Impressions":     "sum",
    "Facebook_Impressions":    "sum",
    "Instagram_Impressions":   "sum",
    "Print_Readership":        "sum",
    "Radio_Listenership":      "sum",
    "Trade_Spend":             "sum",
    "CPI":                     "mean",
    "GDP_Growth":              "mean",
    "Festival_Index":          "mean",
    "Rainfall_Index":          "mean",
}
df_nat = df_raw.groupby("Week", as_index=False).agg(agg)
df_nat = df_nat.rename(columns={
    "Week":                 "date",
    "Sales_Value":          "sales",
    "TV_Impressions":       "tv",
    "YouTube_Impressions":  "youtube",
    "Facebook_Impressions": "facebook",
    "Instagram_Impressions":"instagram",
    "Print_Readership":     "print_media",
    "Radio_Listenership":   "radio",
    "Trade_Spend":          "trade_spend",
    "CPI":                  "cpi",
    "GDP_Growth":           "gdp_growth",
    "Festival_Index":       "festival",
    "Rainfall_Index":       "rainfall",
})
df_nat["date"] = df_nat["date"].dt.strftime("%Y-%m-%d")
df_nat.to_csv("data/processed/mmm_national_weekly.csv", index=False)

print(f"National weekly data: {len(df_nat)} rows × {df_nat.shape[1]} cols")
print(f"Date range: {df_nat['date'].min()} → {df_nat['date'].max()}")
print("\nMedia channel descriptive stats:")
media_cols = ["tv", "youtube", "facebook", "instagram", "print_media", "radio"]
print(df_nat[media_cols].describe().round(0).to_string())


## 2. Define Robyn Input Variables

In [ ]:

ROBYN_ITERATIONS = 200
ROBYN_TRIALS = 1
ROBYN_CORES = 4

ROBYN_SCRIPT = f"""
# =============================================================================
# Robyn MMM — Indian FMCG National Weekly
# =============================================================================

.libPaths(c("C:/Rlibs", .libPaths()))
suppressPackageStartupMessages({{
  library(Robyn)
  library(reticulate)
  library(data.table)
  library(dplyr)
  library(ggplot2)
  library(jsonlite)
}})

set.seed(42)
options(warn = -1)

python_exe <- "C:/Users/BIPLOB GON/.local/bin/python3.14.exe"
Sys.setenv(RETICULATE_PYTHON = python_exe)
reticulate::use_python(python_exe, required = FALSE)

dt <- fread("data/processed/mmm_national_weekly.csv")
dt[, date := as.Date(date)]
cat("Rows:", nrow(dt), "| Cols:", ncol(dt), "\n")
cat("Date range:", format(min(dt$date)), "to", format(max(dt$date)), "\n")

data("dt_prophet_holidays", package = "Robyn")

InputCollect <- robyn_inputs(
  dt_input          = dt,
  dt_holidays       = dt_prophet_holidays,
  date_var          = "date",
  dep_var           = "sales",
  dep_var_type      = "revenue",
  prophet_vars      = c("trend", "season", "holiday"),
  prophet_country   = "IN",
  context_vars      = c("cpi", "gdp_growth", "festival", "trade_spend"),
  paid_media_spends = c("tv", "youtube", "facebook", "instagram",
                        "print_media", "radio"),
  paid_media_vars   = c("tv", "youtube", "facebook", "instagram",
                        "print_media", "radio"),
  window_start      = "2022-07-04",
  window_end        = "2025-06-23",
  adstock           = "geometric"
)

hyperparameters <- list(
  tv_alphas          = c(0.5, 3.0),
  tv_gammas          = c(0.3, 1.0),
  tv_thetas          = c(0.50, 0.85),
  youtube_alphas     = c(0.5, 3.0),
  youtube_gammas     = c(0.3, 1.0),
  youtube_thetas     = c(0.20, 0.55),
  facebook_alphas    = c(0.5, 3.0),
  facebook_gammas    = c(0.3, 1.0),
  facebook_thetas    = c(0.15, 0.50),
  instagram_alphas   = c(0.5, 3.0),
  instagram_gammas   = c(0.3, 1.0),
  instagram_thetas   = c(0.10, 0.45),
  print_media_alphas = c(0.5, 3.0),
  print_media_gammas = c(0.3, 1.0),
  print_media_thetas = c(0.30, 0.65),
  radio_alphas       = c(0.5, 3.0),
  radio_gammas       = c(0.3, 1.0),
  radio_thetas       = c(0.25, 0.60),
  train_size         = c(0.5, 0.8)
)

InputCollect <- robyn_inputs(
  InputCollect    = InputCollect,
  hyperparameters = hyperparameters
)

cat("\n=== Running Nevergrad (iterations={ROBYN_ITERATIONS}, trials={ROBYN_TRIALS}, cores={ROBYN_CORES}) ===\n")
OutputModels <- robyn_run(
  InputCollect = InputCollect,
  cores        = {ROBYN_CORES},
  iterations   = {ROBYN_ITERATIONS},
  trials       = {ROBYN_TRIALS},
  outputs      = FALSE
)
cat("=== Optimisation complete ===\n")

plot_dir <- "outputs/figures/robyn"
dir.create(plot_dir, showWarnings = FALSE, recursive = TRUE)

OutputCollect <- robyn_outputs(
  InputCollect  = InputCollect,
  OutputModels  = OutputModels,
  pareto_fronts = "auto",
  plot_folder   = plot_dir,
  csv_out       = "all"
)

pareto_df <- OutputCollect$allPareto
best_model <- pareto_df$solID[which.min(pareto_df$decomp.rssd)]

tryCatch({{
  robyn_onepagers(
    InputCollect  = InputCollect,
    OutputCollect = OutputCollect,
    select_model  = best_model,
    plot_folder   = plot_dir
  )
}}, error = function(e) cat("One-pager warning:", e$message, "\n"))

AllocatorCollect <- tryCatch(
  robyn_allocator(
    InputCollect       = InputCollect,
    OutputCollect      = OutputCollect,
    select_model       = best_model,
    scenario           = "max_response",
    total_budget       = 1000000,
    channel_constr_low = 0.1,
    channel_constr_up  = 0.9,
    plot_folder        = plot_dir
  ),
  error = function(e) {{ cat("Allocator warning:", e$message, "\n"); NULL }}
)

alloc_csv_path <- "outputs/models/robyn_allocation.csv"
if (!is.null(AllocatorCollect)) {{
  tryCatch({{
    alloc_dt <- AllocatorCollect$dt_optimOut
    fwrite(alloc_dt, alloc_csv_path)
  }}, error = function(e) cat("Alloc table warning:", e$message, "\n"))
}}

roas_dt <- OutputCollect$xDecompAgg[solID == best_model]
fwrite(roas_dt, "outputs/models/robyn_roas.csv")

robyn_subfolders <- list.dirs(plot_dir, recursive = FALSE)
actual_plot_dir  <- if (length(robyn_subfolders) > 0)
                       tail(sort(robyn_subfolders), 1) else plot_dir

meta <- list(
  selected_model  = best_model,
  n_pareto_models = nrow(pareto_df),
  plot_dir        = actual_plot_dir,
  nrmse           = round(pareto_df$nrmse[pareto_df$solID == best_model], 4),
  rssd            = round(pareto_df$decomp.rssd[pareto_df$solID == best_model], 4),
  rsq_train       = round(pareto_df$rsq_train[pareto_df$solID == best_model], 4),
  robyn_version   = as.character(packageVersion("Robyn")),
  iterations      = {ROBYN_ITERATIONS},
  trials          = {ROBYN_TRIALS}
)
write_json(meta, "outputs/models/robyn_meta.json", auto_unbox = TRUE)
cat("Selected model:", best_model, "\n")
cat("Metadata saved to outputs/models/robyn_meta.json\n")
"""

script_path = "robyn_run.R"
with open(script_path, "w", encoding="utf-8") as f:
    f.write(ROBYN_SCRIPT.lstrip())

print(f"R script written → {script_path}  ({Path(script_path).stat().st_size} bytes)")
print(f"\nNotebook-safe search budget: iterations={ROBYN_ITERATIONS}, trials={ROBYN_TRIALS}, cores={ROBYN_CORES}")
print("--- Hyperparameter bounds (decay centred on NB03 estimates) ---")
decay_prior = {"tv": 0.70, "youtube": 0.40, "facebook": 0.35,
               "instagram": 0.30, "print_media": 0.50, "radio": 0.45}
for ch, d in decay_prior.items():
    lo, hi = max(0.05, d - 0.20), min(0.95, d + 0.20)
    print(f"  {ch:15s}  theta ∈ [{lo:.2f}, {hi:.2f}]  (NB03 point estimate: {d})")


## 3. Set Hyperparameter Bounds

In [ ]:

# ── Execute Robyn R script, but reuse cached outputs if they already exist ────
cache_files = [
    Path("outputs/models/robyn_meta.json"),
    Path("outputs/models/robyn_roas.csv"),
]

if all(path.exists() for path in cache_files):
    print("Using cached Robyn outputs:")
    for path in cache_files:
        print(" ", path)
else:
    print(f"Starting Robyn optimisation …  (iterations={ROBYN_ITERATIONS} × trials={ROBYN_TRIALS} on {ROBYN_CORES} cores)")
    result = subprocess.run(
        [R_EXE, "robyn_run.R"],
        capture_output=True,
        text=True,
        env=R_ENV,
        timeout=7200
    )

    stdout_tail = result.stdout[-4000:] if len(result.stdout) > 4000 else result.stdout
    print(stdout_tail)

    if result.returncode != 0:
        print("\n─── STDERR ───")
        print(result.stderr[-2000:] if len(result.stderr) > 2000 else result.stderr)
        raise RuntimeError(f"robyn_run.R exited with code {result.returncode}")

    print(f"\n✓  R script finished  (return code {result.returncode})")
    with open("outputs/models/robyn_run.log", "w", encoding="utf-8") as f:
        f.write(result.stdout)
    print("Full log → outputs/models/robyn_run.log")


## 4. Run Nevergrad Evolutionary Optimisation

In [ ]:

# ── Load model metadata ───────────────────────────────────────────────────────
with open("outputs/models/robyn_meta.json") as f:
    meta = json.load(f)

print("Selected model :", meta["selected_model"])
print(f"  NRMSE        : {meta['nrmse']:.4f}")
print(f"  Decomp RSSD  : {meta['rssd']:.4f}")
print(f"  R² (train)   : {meta['rsq_train']:.4f}")
print(f"Pareto models  : {meta['n_pareto_models']}")

# ── Read Pareto CSVs from Robyn output directory ──────────────────────────────
pareto_csv = sorted(Path(meta["plot_dir"]).glob("*pareto_aggregated*.csv"))
if not pareto_csv:
    # Fallback: search robyn folder recursively
    pareto_csv = sorted(Path("outputs/figures/robyn").rglob("*pareto_aggregated*.csv"))

if pareto_csv:
    df_pareto = pd.read_csv(pareto_csv[-1])
    print(f"\nPareto CSV: {pareto_csv[-1].name}  ({len(df_pareto)} rows)")

    # ── Pareto frontier scatter: NRMSE vs RSSD ───────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 5))
    sc = ax.scatter(df_pareto["nrmse"], df_pareto["decomp.rssd"],
                    c=df_pareto.get("rsq_train", 0.5),
                    cmap="RdYlGn", s=60, alpha=0.75, edgecolors="k", linewidths=0.4)
    plt.colorbar(sc, ax=ax, label="R² (train)")

    # Highlight selected model
    sel = df_pareto[df_pareto["solID"] == meta["selected_model"]]
    if not sel.empty:
        ax.scatter(sel["nrmse"], sel["decomp.rssd"],
                   marker="*", s=300, color="royalblue", zorder=5,
                   label=f"Selected: {meta['selected_model']}")
        ax.legend(fontsize=9)

    ax.set_xlabel("NRMSE (lower = better fit)", fontsize=11)
    ax.set_ylabel("Decomp. RSSD (lower = less Business Error)", fontsize=11)
    ax.set_title("Robyn Pareto Frontier — NRMSE vs. RSSD\n"
                 "(Colour = R², Star = selected model)", fontsize=12)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("outputs/figures/05_robyn_pareto_frontier.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved → outputs/figures/05_robyn_pareto_frontier.png")
else:
    print("Pareto CSV not found — check outputs/figures/robyn/ directory")


## 5. Pareto Frontier Exploration

In [ ]:

# ── Find and display the Robyn one-pager PNG ─────────────────────────────────
onepager_paths = sorted(Path(meta["plot_dir"]).glob(f"*{meta['selected_model']}*onepager*.png"))
if not onepager_paths:
    onepager_paths = sorted(Path("outputs/figures/robyn").rglob("*onepager*.png"))
if not onepager_paths:
    onepager_paths = sorted(Path("outputs/figures/robyn").rglob("*.png"))

if onepager_paths:
    img_path = str(onepager_paths[0])
    print(f"One-pager: {img_path}")
    fig, ax = plt.subplots(figsize=(16, 10))
    ax.imshow(mpimg.imread(img_path))
    ax.axis("off")
    ax.set_title(f"Robyn One-Pager — Model {meta['selected_model']}", fontsize=13, pad=8)
    plt.tight_layout()
    plt.savefig("outputs/figures/05_robyn_onepager.png", dpi=120, bbox_inches="tight")
    plt.show()
else:
    print("No one-pager PNG found in", meta["plot_dir"])
    print("Available PNGs:")
    for p in sorted(Path("outputs/figures/robyn").rglob("*.png"))[:10]:
        print(" ", p)


## 6. One-Pager Summary Chart

In [ ]:

# ── Budget allocator results ──────────────────────────────────────────────────
alloc_path = Path("outputs/models/robyn_allocation.csv")
if alloc_path.exists():
    df_alloc = pd.read_csv(alloc_path)
    print("Budget Allocator — Max Response  (total budget = ₹1,000,000)")
    print(df_alloc.to_string(index=False))

    # Bar chart: current vs optimised spend
    channels = df_alloc.columns[df_alloc.columns.str.contains("channel|rn|variable",
                                                                case=False)].tolist()
    if not channels:
        channels = df_alloc.columns[:1].tolist()
    ch_col = channels[0]

    spend_cols = [c for c in df_alloc.columns if "spend" in c.lower() or "alloc" in c.lower()]
    if len(spend_cols) >= 2:
        fig, ax = plt.subplots(figsize=(9, 5))
        x = np.arange(len(df_alloc))
        w = 0.38
        ax.bar(x - w/2, df_alloc[spend_cols[0]] / 1000, w, label=spend_cols[0], alpha=0.8)
        ax.bar(x + w/2, df_alloc[spend_cols[1]] / 1000, w, label=spend_cols[1], alpha=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(df_alloc[ch_col].tolist(), rotation=30, ha="right")
        ax.set_ylabel("Spend (₹ thousands)")
        ax.set_title("Robyn Budget Allocator — Current vs. Optimised Spend")
        ax.legend()
        ax.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        plt.savefig("outputs/figures/05_robyn_budget_allocation.png", dpi=150, bbox_inches="tight")
        plt.show()
    else:
        print("Column names:", df_alloc.columns.tolist())
else:
    print("Allocation CSV not found. The allocator may have been skipped.")
    print("Check outputs/models/ for any allocation files.")


## 7. Budget Allocator

In [ ]:

# ── Robyn ROAS vs. Bayesian ROAS comparison ────────────────────────────────────
roas_path = Path("outputs/models/robyn_roas.csv")
bayes_path = Path("outputs/models/bayesian_roas.parquet")

if roas_path.exists():
    df_roas_robyn = pd.read_csv(roas_path)
    print("Robyn xDecompAgg columns:", df_roas_robyn.columns.tolist())

    # Identify the channel/ROAS columns in Robyn's output format
    rn_col = "rn" if "rn" in df_roas_robyn.columns else df_roas_robyn.columns[0]
    roas_col = next((c for c in df_roas_robyn.columns if "roas" in c.lower()), None)
    spend_col = next((c for c in df_roas_robyn.columns if "spend" in c.lower()), None)
    response_col = next((c for c in df_roas_robyn.columns
                          if "response" in c.lower() or "effect" in c.lower()), None)

    media_channels = ["tv", "youtube", "facebook", "instagram", "print_media", "radio"]
    df_media = df_roas_robyn[df_roas_robyn[rn_col].isin(media_channels)].copy()

    if roas_col and not df_media.empty:
        df_media = df_media[[rn_col, roas_col]].rename(
            columns={rn_col: "channel", roas_col: "robyn_roas"}).set_index("channel")
        print("\nRobyn ROAS by channel:")
        print(df_media.round(3).to_string())

        # ── Compare with Bayesian ROAS ──────────────────────────────────────
        comparison = df_media.copy()
        if bayes_path.exists():
            df_bayes = pd.read_parquet(bayes_path)
            print("\nBayesian ROAS columns:", df_bayes.columns.tolist())
            # Try to align on channel name
            if "channel" in df_bayes.columns and "roas_mean" in df_bayes.columns:
                df_bayes_sel = df_bayes[
                    df_bayes["channel"].isin(media_channels)
                ][["channel", "roas_mean"]].set_index("channel")
                comparison = comparison.join(df_bayes_sel.rename(
                    columns={"roas_mean": "bayesian_roas"}), how="left")

        print("\nROAS comparison — Robyn vs. Bayesian:")
        print(comparison.round(3).to_string())

        # ── ROAS bar chart ──────────────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(9, 5))
        x = np.arange(len(comparison))
        w = 0.35 if "bayesian_roas" in comparison.columns else 0.6
        ax.bar(x - w/2 if "bayesian_roas" in comparison.columns else x,
               comparison["robyn_roas"], w,
               label="Robyn ROAS", color="steelblue", alpha=0.85)
        if "bayesian_roas" in comparison.columns:
            ax.bar(x + w/2, comparison["bayesian_roas"].fillna(0), w,
                   label="Bayesian ROAS", color="darkorange", alpha=0.85)
        ax.set_xticks(x)
        ax.set_xticklabels(comparison.index.tolist(), rotation=30, ha="right")
        ax.set_ylabel("ROAS (₹ sales per ₹ impressions)")
        ax.set_title("ROAS: Robyn (Evolutionary) vs. Bayesian MMM")
        ax.legend(); ax.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        plt.savefig("outputs/figures/05_robyn_roas_comparison.png", dpi=150, bbox_inches="tight")
        plt.show()
        print("\nSaved → outputs/figures/05_robyn_roas_comparison.png")
    else:
        print("ROAS column not found. Available cols:", df_roas_robyn.columns.tolist())
else:
    print("robyn_roas.csv not found — check R script execution")


## 8. Export ROAS for Python Comparison

In [ ]:

# ── NB05 Summary ──────────────────────────────────────────────────────────────
print("=" * 60)
print("  NB05 — Robyn MMM Summary")
print("=" * 60)
with open("outputs/models/robyn_meta.json") as f:
    meta = json.load(f)

print(f"\nFramework   : Meta Robyn {meta.get('robyn_version','3.12.1')}")
print(f"Optimiser   : Nevergrad evolutionary (iterations={meta.get('iterations')}, trials={meta.get('trials')})")
print(f"Adstock     : Geometric  |  Saturation: Hill")
print(f"Country     : India (IN) — Prophet holiday effects")
print(f"\nSelected model : {meta['selected_model']}")
print(f"  NRMSE        : {meta['nrmse']:.4f}  (normalised RMSE)")
print(f"  Decomp RSSD  : {meta['rssd']:.4f}  (business constraint score)")
print(f"  R² (train)   : {meta['rsq_train']:.4f}")
print(f"  Pareto set   : {meta['n_pareto_models']} models")

print("\nOutputs:")
for f_path in ["outputs/models/robyn_roas.csv",
               "outputs/models/robyn_allocation.csv",
               "outputs/figures/05_robyn_pareto_frontier.png",
               "outputs/figures/05_robyn_onepager.png",
               "outputs/figures/05_robyn_roas_comparison.png"]:
    status = "✓" if Path(f_path).exists() else "✗"
    print(f"  {status}  {f_path}")
